In [1]:
import sys
import time
import numpy as np
import pandas as pd
from pathlib import Path

# 1. Path Resolution: Ensure we are importing the local fuzzycore package
# Adjust the '.parent' chain if your notebook is deeper in the directory structure
fuzzycore_root = str(Path.cwd().parent)
if fuzzycore_root not in sys.path:
    sys.path.append(fuzzycore_root)

from fuzzycore import eos
from fuzzycore.utils import generate_gaussian_z_profile

print("🟢 [1] Environment and imports loaded successfully.", flush=True)

# 2. ISOLATE THE RAW DATA LOADER
print("\n🟢 [2] Triggering load_all_raw_data()...", flush=True)
t0 = time.time()
try:
    raw_tables = eos.load_all_raw_data()
    print(f"   ✅ Finished loading all raw data from disk in {time.time()-t0:.2f} seconds.", flush=True)
    
    print("\n   📊 Loaded Tables Summary:")
    for k, v in raw_tables.items():
        print(f"      - {k}: shape {v.shape}", flush=True)
except Exception as e:
    print(f"   ❌ CRASHED DURING FILE I/O: {e}", flush=True)
    sys.exit(1)

# 3. ISOLATE THE CORE INTERPOLATOR
print("\n🟢 [3] Testing get_rock_interpolator()...", flush=True)
t0 = time.time()
try:
    rock_interp = eos.get_rock_interpolator()
    print(f"   ✅ Success! Core interpolator built in {time.time()-t0:.2f} seconds.", flush=True)
except Exception as e:
    print(f"   ❌ CRASHED DURING CORE INTERPOLATION: {e}", flush=True)
    sys.exit(1)

# 4. ISOLATE THE Z-PROFILE GENERATOR
print("\n🟢 [4] Generating flat 1-layer Z-profile (Sharp Core)...", flush=True)
try:
    flat_z = generate_gaussian_z_profile(sigma=0.0, z_base=0.02)
    print(f"   ✅ Z profile generated: {flat_z}", flush=True)
except Exception as e:
    print(f"   ❌ CRASHED DURING Z-PROFILE GENERATION: {e}", flush=True)
    sys.exit(1)

# 5. ISOLATE THE FLUID INTERPOLATOR
print("\n🟢 [5] Testing generate_fluid_interpolators() for 1 layer...", flush=True)
t0 = time.time()
try:
    fluid_stack = eos.generate_fluid_interpolators(flat_z, y_ratio=0.26)
    print(f"   ✅ Success! Fluid stack built in {time.time()-t0:.2f} seconds.", flush=True)
except Exception as e:
    print(f"   ❌ CRASHED DURING FLUID INTERPOLATION: {e}", flush=True)

print("\n🏁 ALL EOS TESTS COMPLETE.", flush=True)

🟢 [1] Environment and imports loaded successfully.

🟢 [2] Triggering load_all_raw_data()...
--- Loading Raw EOS Tables (From Disk) ---
  > Loading Hydrogen...
  > Loading Helium...
  > Loading Water...
  > Loading Rock...
  > Loading Iron...
   ✅ Finished loading all raw data from disk in 0.78 seconds.

   📊 Loaded Tables Summary:
      - H: shape (53259, 4)
      - He: shape (53208, 4)
      - H2O: shape (8303, 4)
      - Rock: shape (25050, 4)
      - Iron: shape (249983, 4)

🟢 [3] Testing get_rock_interpolator()...
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
   ✅ Success! Core interpolator built in 8.84 seconds.

🟢 [4] Generating flat 1-layer Z-profile (Sharp Core)...
   ✅ Z profile generated: [0.02]

🟢 [5] Testing generate_fluid_interpolators() for 1 layer...
   ✅ Success! Fluid 

In [2]:
import sys
import time
from pathlib import Path
import numpy as np

# 1. Path Resolution: Ensure we are importing the local fuzzycore package
fuzzycore_root = str(Path.cwd().parent)
if fuzzycore_root not in sys.path:
    sys.path.append(fuzzycore_root)

from fuzzycore import constants as c
from fuzzycore import solver
from fuzzycore import utils

# =============================================================================
# REPLICATING THE SUB-NEPTUNE SETUP
# =============================================================================
# We mimic the exact dictionary the Coupler passes to the solver under the hood.
target_mass_kg = 8.0 * c.M_EARTH

params_sharp = {
    "P_surf": 1.0,                             # from p_bottom_bar
    "T_surf": 500.0,                           # from T_irr
    "T_int": 100.0,                            # from T_int
    "M_core": 7.9 * c.M_EARTH,                 # core_mass_earth mapped to kg
    "iron_fraction": 0.33,                     # Standard rock/iron ratio
    "z_profile": np.array([0.02]),             # Flat 1-layer profile (sigma=0.0)
    "Y_ratio": 0.26,                           # Standard Helium ratio
    "debug": True
}

print("🚀 Launching exact Sharp Core Sub-Neptune solver (unmodified)...", flush=True)
print("⏳ EXPECTED BEHAVIOR: The console will go completely silent here.", flush=True)
print("   It is currently running the 15-point global scan with microscopic spatial steps.", flush=True)

t0 = time.time()
dummy_lock = utils.DummyLock()

try:
    # Trigger the exact function that caused the "freeze"
    final_model = solver.solve_structure(
        target_val=target_mass_kg,
        params=params_sharp,
        mode='mass',
        trial_id='sub_neptune_test',
        csv_file='dummy_test.csv',
        write_lock=dummy_lock
    )
    
    elapsed = time.time() - t0
    print(f"\n✅ SOLVER FINISHED in {elapsed:.1f} seconds!")
    if final_model:
        print(f"   Final Planet Radius: {final_model['R'][-1] / c.R_EARTH:.2f} R_Earth")
except Exception as e:
    print(f"\n❌ ERRORED OUT: {e}")

🚀 Launching exact Sharp Core Sub-Neptune solver (unmodified)...
⏳ EXPECTED BEHAVIOR: The console will go completely silent here.
   It is currently running the 15-point global scan with microscopic spatial steps.
Trial sub_neptune_test: Global scan initiated
  [Solver] logPc 6.50: Integration returned None
[START] Pc=1.00e+07 -> P_int=2.42e+06 -> T_int=16529.2 K
[SUCCESS] M=0.205 Mj
  [Debug] sub_neptune_test logPc: 7.00 -> Mass: 0.205 Mj
[START] Pc=3.16e+07 -> P_int=1.75e+07 -> T_int=30825.9 K
[SUCCESS] M=0.796 Mj
  [Debug] sub_neptune_test logPc: 7.50 -> Mass: 0.796 Mj
[START] Pc=1.00e+08 -> P_int=7.22e+07 -> T_int=47753.8 K
[SUCCESS] M=1.756 Mj
  [Debug] sub_neptune_test logPc: 8.00 -> Mass: 1.756 Mj
[START] Pc=3.16e+08 -> P_int=2.58e+08 -> T_int=78732.5 K
[SUCCESS] M=3.341 Mj
  [Debug] sub_neptune_test logPc: 8.50 -> Mass: 3.341 Mj
[START] Pc=1.00e+09 -> P_int=8.73e+08 -> T_int=123060.3 K
[SUCCESS] M=5.823 Mj
  [Debug] sub_neptune_test logPc: 9.00 -> Mass: 5.823 Mj
[START] Pc=3.16e